# CURE-OR++ Open-Weight VLM Real-Transfer GPU Pilot

This notebook runs an open-weight VLM over the CURE-OR++ real-transfer v0.2
prompt pack. It is intended to produce a reproducible VLM baseline without
paid frontier API calls.

Default model: `HuggingFaceTB/SmolVLM2-500M-Video-Instruct`.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import pandas as pd

os.environ.setdefault('HF_HOME', '/kaggle/working/hf_home')

DATASET_CANDIDATES = [
    Path('/kaggle/input/cure-or-pp-vlm-real-transfer-v02-private'),
    Path('/kaggle/input/cure-or-plus-plus-vlm-real-transfer-v02-private'),
]
DATA_ROOT = next((path for path in DATASET_CANDIDATES if (path / 'reports/vlm_api_track_v01_prompt_pack.jsonl').exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Attach the private CURE-OR++ VLM real-transfer dataset to this notebook.')

DATA_ROOT


In [ ]:
%pip install -q num2words


In [ ]:
PROMPT_PACK = DATA_ROOT / 'reports/vlm_api_track_v01_prompt_pack.jsonl'
RUNNER = DATA_ROOT / 'scripts/run_hf_vlm.py'
EVALUATOR = DATA_ROOT / 'scripts/evaluate_vlm_response_pack.py'

MODEL_ID = 'HuggingFaceTB/SmolVLM2-500M-Video-Instruct'
RUN_LIMIT = 210  # set to 5 for smoke testing
RESPONSES = Path('/kaggle/working/vlm_responses_smolvlm2_500m.jsonl')

cmd = [
    sys.executable,
    str(RUNNER),
    '--prompt-pack', str(PROMPT_PACK),
    '--model', MODEL_ID,
    '--output', str(RESPONSES),
    '--cache-dir', '/kaggle/working/vlm_response_cache',
    '--limit', str(RUN_LIMIT),
    '--force',
    '--max-tokens', '8',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True, cwd=str(DATA_ROOT))


In [ ]:
MODEL_SUMMARY = Path('/kaggle/working/vlm_smolvlm2_model_summary.csv')
RECIPE_TABLE = Path('/kaggle/working/vlm_smolvlm2_recipe_table.csv')
LABEL_TABLE = Path('/kaggle/working/vlm_smolvlm2_label_table.csv')
AUDIT_TABLE = Path('/kaggle/working/vlm_smolvlm2_audit.csv')

cmd = [
    sys.executable,
    str(EVALUATOR),
    '--prompt-pack', str(PROMPT_PACK),
    '--responses', str(RESPONSES),
    '--model-summary', str(MODEL_SUMMARY),
    '--recipe-table', str(RECIPE_TABLE),
    '--label-table', str(LABEL_TABLE),
    '--audit-table', str(AUDIT_TABLE),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True, cwd=str(DATA_ROOT))


In [ ]:
pd.read_csv(MODEL_SUMMARY)


In [ ]:
pd.read_csv(RECIPE_TABLE)


In [ ]:
pd.read_csv(LABEL_TABLE).sort_values(['real_accuracy', 'label']).head(20)
